# Batch 2 — Preprocessing (Imbalance + Redundancy)

**What Batch 1 found that motivates this batch:**
1. Even the best models (CatBoost/XGBoost/LightGBM, ~0.80 PR-AUC) only reach
   **~60% recall** at the default threshold — 40% of fraud cases are missed.
   No sampling or class weighting was used in Batch 1 at all.
2. CatBoost's feature importances showed the model leans on `Tax_Gap`
   (~55%) and `Declared_Tax` (~18%) almost exclusively — the
   `Tax_Gap`/`Expected_Tax`/`Declared_Tax`/`Taxable_Income` cluster is ~83%
   of total importance, even though EDA showed several of these columns are
   exact mathematical duplicates of each other.

This notebook reuses the **exact same fixed train/val/test split** and base
encoders from `Models_Batch_1/processing.ipynb` (loaded directly — no
re-fitting, no re-randomizing), and adds two things Batch 1 didn't have:

- **Reduced feature sets** with the mathematically redundant columns
  removed, sized by an actual VIF analysis (not guesswork)
- **Imbalance-handling building blocks** (samplers) — the samplers
  themselves are *not* applied here. Per the project plan, sampling must
  happen only inside training folds, so `training.ipynb` will wrap each
  sampler in an `imblearn` pipeline that resamples only the training portion
  of each CV fold, never validation or test.


In [1]:
import numpy as np
import pandas as pd
import joblib
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
RANDOM_STATE = 42


## 1. Paths & Load Batch 1 Artifacts

Reusing Batch 1's fixed split and fitted encoders directly — no re-fitting.

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'Models_Batch_2' else Path.cwd()
BATCH1_DIR = PROJECT_ROOT / 'Models_Batch_1'
BATCH_DIR = PROJECT_ROOT / 'Models_Batch_2'
ARTIFACTS_DIR = BATCH_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
(ARTIFACTS_DIR / 'models').mkdir(exist_ok=True)

data1 = joblib.load(BATCH1_DIR / 'artifacts' / 'batch1_data.joblib')
print("Loaded Batch 1 artifacts. Train/val/test shapes:",
      data1['X_train_tree'].shape, data1['X_val_tree'].shape, data1['X_test_tree'].shape)
print(f"Fraud rate — train: {data1['y_train'].mean():.4f}, val: {data1['y_val'].mean():.4f}, test: {data1['y_test'].mean():.4f}")


Loaded Batch 1 artifacts. Train/val/test shapes: (35000, 40) (7500, 40) (7500, 40)
Fraud rate — train: 0.1072, val: 0.1072, test: 0.1072


## 2. VIF Analysis — Full Feature Set

`statsmodels`' `variance_inflation_factor` requires an intercept column to
be present in the design matrix — without it, VIF is computed relative to
the origin rather than the mean, which produces spuriously huge values for
any variable whose mean is far from zero (e.g. `Annual_Revenue`, mean
~1.5M). We add a constant via `sm.add_constant()` before computing VIF.


In [3]:
def compute_vif(df):
    df_c = sm.add_constant(df)
    vif_data = pd.DataFrame()
    vif_data['feature'] = df_c.columns
    vif_data['VIF'] = [variance_inflation_factor(df_c.values, i) for i in range(df_c.shape[1])]
    return vif_data[vif_data['feature'] != 'const'].sort_values('VIF', ascending=False).reset_index(drop=True)

X_train_full = data1['X_train_tree']
numeric_and_ordinal = data1['numeric_cols'] + ['Industry_Risk_ord']

vif_full = compute_vif(X_train_full[numeric_and_ordinal].astype(float))
vif_full


C:\Users\Ali Ahmed\AppData\Local\Programs\Python\Python38\lib\site-packages\statsmodels\regression\linear_model.py:1782: RuntimeWarning: divide by zero encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
C:\Users\Ali Ahmed\AppData\Local\Programs\Python\Python38\lib\site-packages\statsmodels\stats\outliers_influence.py:198: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,feature,VIF
0,Profit_Margin,inf
1,Annual_Revenue,inf
2,Annual_Expenses,inf
3,Expense_Ratio,inf
4,Net_Profit,9.007199e+15
5,VAT_Collected,3.002400e+15
6,Taxable_Income,2.251800e+15
7,VAT_Paid,1.801440e+15
8,Expected_Tax,1.501200e+15
9,Declared_Tax,7.505999e+14


**Interpretation:** `Annual_Revenue`, `Annual_Expenses`, `Expense_Ratio`,
`Profit_Margin` show `inf` VIF, and the rest of the financial block
(`Net_Profit`, `VAT_Collected`, `Taxable_Income`, `VAT_Paid`, `Expected_Tax`,
`Declared_Tax`, `Tax_Gap`) shows astronomically large values. This is
**mathematically expected**, not a data problem — confirmed in
`Cleaning.ipynb`/`EDA.ipynb`:

- `VAT_Collected` = 14% × `Annual_Revenue` (exact)
- `VAT_Paid` = 14% × `Annual_Expenses` (exact)
- `Expected_Tax` = 22.5% × `Taxable_Income` (exact)
- `Net_Profit` = `Annual_Revenue` − `Annual_Expenses` (exact)
- `Expense_Ratio` = `Annual_Expenses` / `Annual_Revenue`, and
  `Profit_Margin` = 1 − `Expense_Ratio` (exact)
- `Tax_Gap` = `Expected_Tax` − `Declared_Tax` (exact)

Every behavioral/categorical feature (`Previous_Violations`,
`Invoice_Mismatch`, `Missing_Documents`, `Cash_Transactions_Percentage`,
`Industry_Risk`, `Previous_Audits`, `Late_Payments`, `Years_in_Business`)
has VIF ≈ 1.0 — completely independent, exactly as EDA suggested. These are
kept in every feature set below, per the project plan.


## 3. Reduced Feature Set v1 — Remove Exact Mathematical Duplicates

To resolve a linear redundancy cluster, only *one* column needs to be
dropped per group of exactly-dependent columns (removing all but one loses
no information). We keep the more behaviorally meaningful / interpretable
member of each pair:

| Redundant group | Keep | Drop | Why keep this one |
|---|---|---|---|
| `VAT_Collected` = 14%×`Annual_Revenue` | `Annual_Revenue` | `VAT_Collected` | Revenue is the more fundamental quantity |
| `VAT_Paid` = 14%×`Annual_Expenses` | `Annual_Expenses` | `VAT_Paid` | same reasoning |
| `Expected_Tax`, `Taxable_Income`, `Declared_Tax`, `Tax_Gap` (rank-2 cluster: `Tax_Gap`=`Expected_Tax`−`Declared_Tax`, `Expected_Tax`=22.5%×`Taxable_Income`) | `Declared_Tax`, `Tax_Gap` | `Expected_Tax`, `Taxable_Income` | `Tax_Gap` is the strongest predictor found in EDA/Batch 1 and is literally what auditors compare; `Declared_Tax` is the taxpayer's actual behavior. Together they're non-redundant; `Expected_Tax`/`Taxable_Income` add nothing beyond what `Tax_Gap`+`Declared_Tax` already encode. |
| `Net_Profit` = `Annual_Revenue`−`Annual_Expenses`; `Expense_Ratio`, `Profit_Margin` (both exact functions of Revenue & Expenses) | *(nothing — see below)* | `Net_Profit`, `Expense_Ratio`, `Profit_Margin` | All three are exact arithmetic of `Annual_Revenue`/`Annual_Expenses`, which are kept |

This drops **7 columns**: `VAT_Collected`, `VAT_Paid`, `Expected_Tax`,
`Taxable_Income`, `Net_Profit`, `Expense_Ratio`, `Profit_Margin`.


In [4]:
EXACT_DUP_DROPS = ['VAT_Collected', 'VAT_Paid', 'Expected_Tax', 'Taxable_Income',
                    'Net_Profit', 'Expense_Ratio', 'Profit_Margin']

reduced_v1_numeric = [c for c in numeric_and_ordinal if c not in EXACT_DUP_DROPS]
vif_v1 = compute_vif(X_train_full[reduced_v1_numeric].astype(float))
vif_v1


,feature,VIF
0,Annual_Revenue,1207.860592
1,Annual_Expenses,609.001485
2,Declared_Tax,224.696365
3,Revenue_per_Employee,2.232170
4,Tax_Gap,2.120138
5,Employee_Count,1.347677
6,Previous_Violations,1.002454
7,Industry_Risk_ord,1.002354
8,Invoice_Mismatch,1.002075
9,Missing_Documents,1.001603


**This does *not* fully resolve the multicollinearity.** `Annual_Revenue`
(VIF ≈ 1,200), `Annual_Expenses` (VIF ≈ 600), and `Declared_Tax` (VIF ≈ 225)
remain severely inflated — but this time it's *not* an exact formula. It's a
real statistical relationship: for the ~89% of taxpayers who are compliant,
`Declared_Tax` ≈ `Expected_Tax` ≈ 22.5% × `Taxable_Income` ≈ 22.5% × 92.5% ×
(`Annual_Revenue` − `Annual_Expenses`) — so even without the intermediate
columns present, `Declared_Tax` is still *statistically* (not just
formulaically) predictable from `Annual_Revenue` and `Annual_Expenses`
combined, for most of the dataset.

Both EDA and Batch 1's feature importances showed `Annual_Revenue` and
`Annual_Expenses` carry **essentially no standalone fraud signal**
(correlation with `Fraud` of 0.002 and 0.003; near-zero CatBoost
importance) — so unlike the exact-duplicate case above, dropping them isn't
just "no information lost," it's "no *useful* information lost."


## 4. Reduced Feature Set v2 — Also Drop Annual_Revenue / Annual_Expenses

In [5]:
REVENUE_SIZE_DROPS = ['Annual_Revenue', 'Annual_Expenses']
reduced_v2_numeric = [c for c in reduced_v1_numeric if c not in REVENUE_SIZE_DROPS]

vif_v2 = compute_vif(X_train_full[reduced_v2_numeric].astype(float))
vif_v2


,feature,VIF
0,Declared_Tax,1.761190
1,Revenue_per_Employee,1.670431
2,Tax_Gap,1.379527
3,Employee_Count,1.259817
4,Previous_Violations,1.002434
5,Industry_Risk_ord,1.002281
6,Invoice_Mismatch,1.002055
7,Missing_Documents,1.001531
8,Cash_Transactions_Percentage,1.000996
9,Years_in_Business,1.000667


**Fully resolved** — every remaining feature has VIF < 2, well under the
common concern threshold of 5. This is the feature set most useful for a
**linear** model like Logistic Regression, where severe multicollinearity
inflates coefficient variance and makes coefficients unstable/hard to
interpret (though regularization already partially compensates, as seen in
Batch 1's L1-penalized Logistic Regression).

**Important:** VIF is a linear-model diagnostic. Tree-based/boosting models
split on one feature at a time and are mathematically invariant to linear
redundancy among features — Batch 1 showed no overfitting for any model, so
we do **not** expect the reduced sets to meaningfully change tree-based
model performance. Testing them anyway (in `training.ipynb`) is a useful
confirmation of that expectation, not a correction of a real problem for
those models.

Per the project plan, `Previous_Violations`, `Missing_Documents`, and
`Invoice_Mismatch` are untouched in every feature set — their VIF has been
~1.0 throughout, and Batch 1 confirmed they carry real (if individually
modest) signal.


## 5. Build the Three Feature Set Variants

`full` = Batch 1's original feature set (baseline for comparison).
`reduced_v1` = exact-duplicate columns removed.
`reduced_v2` = also removes `Annual_Revenue`/`Annual_Expenses` (VIF-clean).

Built for all four representations already prepared in Batch 1: unscaled
(`tree`), `StandardScaler` (`std`), `RobustScaler` (`rob`), and CatBoost's
native categorical (`cb`) — by simply **subsetting columns** from Batch 1's
already-correctly-fit encoded data. No re-fitting is needed: dropping
columns from a correctly encoded matrix introduces no leakage.


In [6]:
def build_variant(numeric_subset):
    """Return dict of {tree, std, rob} feature frames restricted to numeric_subset + Industry_Risk_ord + one-hot cols."""
    cols_to_keep = lambda X: [c for c in X.columns
                               if c in numeric_subset
                               or c == 'Industry_Risk_ord'
                               or c not in (data1['numeric_cols'] + ['Industry_Risk_ord'])]
    out = {}
    for key in ['tree', 'std', 'rob']:
        out[f'X_train_{key}'] = data1[f'X_train_{key}'][cols_to_keep(data1[f'X_train_{key}'])]
        out[f'X_val_{key}'] = data1[f'X_val_{key}'][cols_to_keep(data1[f'X_val_{key}'])]
        out[f'X_test_{key}'] = data1[f'X_test_{key}'][cols_to_keep(data1[f'X_test_{key}'])]
    return out

# 'full' just points at Batch 1's data directly (no columns dropped)
variants = {
    'full': {f'{split}_{key}': data1[f'{split}_{key}']
             for split in ['X_train', 'X_val', 'X_test'] for key in ['tree', 'std', 'rob']},
    'reduced_v1': build_variant([c for c in reduced_v1_numeric if c != 'Industry_Risk_ord']),
    'reduced_v2': build_variant([c for c in reduced_v2_numeric if c != 'Industry_Risk_ord']),
}

for name, v in variants.items():
    print(f"{name:>12}: tree shape = {v['X_train_tree'].shape}, columns dropped from full = "
          f"{sorted(set(data1['X_train_tree'].columns) - set(v['X_train_tree'].columns))}")


        full: tree shape = (35000, 40), columns dropped from full = []
  reduced_v1: tree shape = (35000, 33), columns dropped from full = ['Expected_Tax', 'Expense_Ratio', 'Net_Profit', 'Profit_Margin', 'Taxable_Income', 'VAT_Collected', 'VAT_Paid']
  reduced_v2: tree shape = (35000, 31), columns dropped from full = ['Annual_Expenses', 'Annual_Revenue', 'Expected_Tax', 'Expense_Ratio', 'Net_Profit', 'Profit_Margin', 'Taxable_Income', 'VAT_Collected', 'VAT_Paid']


## 6. CatBoost Native Feature Sets (per variant)

CatBoost's native categorical frame (`X_train_cb`) only ever contained the
raw numeric/ordinal columns plus `Business_Type`/`Region` as categories
(no one-hot expansion), so we subset it directly by column name.


In [7]:
def build_cb_variant(numeric_subset):
    keep_cols = numeric_subset + ['Industry_Risk', 'Business_Type', 'Region']
    out = {}
    for split in ['X_train_cb', 'X_val_cb', 'X_test_cb']:
        out[split] = data1[split][keep_cols]
    return out

variants['full']['X_train_cb'] = data1['X_train_cb']
variants['full']['X_val_cb'] = data1['X_val_cb']
variants['full']['X_test_cb'] = data1['X_test_cb']

variants['reduced_v1'].update(build_cb_variant([c for c in reduced_v1_numeric if c != 'Industry_Risk_ord']))
variants['reduced_v2'].update(build_cb_variant([c for c in reduced_v2_numeric if c != 'Industry_Risk_ord']))

for name, v in variants.items():
    print(f"{name:>12}: catboost shape = {v['X_train_cb'].shape}, columns = {v['X_train_cb'].columns.tolist()}")


        full: catboost shape = (35000, 23), columns = ['Business_Type', 'Region', 'Years_in_Business', 'Employee_Count', 'Annual_Revenue', 'Annual_Expenses', 'Net_Profit', 'Taxable_Income', 'Expected_Tax', 'Declared_Tax', 'VAT_Collected', 'VAT_Paid', 'Previous_Audits', 'Previous_Violations', 'Late_Payments', 'Industry_Risk', 'Cash_Transactions_Percentage', 'Missing_Documents', 'Invoice_Mismatch', 'Expense_Ratio', 'Profit_Margin', 'Revenue_per_Employee', 'Tax_Gap']
  reduced_v1: catboost shape = (35000, 16), columns = ['Years_in_Business', 'Employee_Count', 'Annual_Revenue', 'Annual_Expenses', 'Declared_Tax', 'Previous_Audits', 'Previous_Violations', 'Late_Payments', 'Cash_Transactions_Percentage', 'Missing_Documents', 'Invoice_Mismatch', 'Revenue_per_Employee', 'Tax_Gap', 'Industry_Risk', 'Business_Type', 'Region']
  reduced_v2: catboost shape = (35000, 14), columns = ['Years_in_Business', 'Employee_Count', 'Declared_Tax', 'Previous_Audits', 'Previous_Violations', 'Late_Payments', 'Cas

## 7. Categorical Feature Indices for SMOTENC

Plain `SMOTE` interpolates between nearest neighbors — applied directly to
one-hot columns, that produces nonsensical fractional values (e.g. 0.3 for
a column that should only ever be 0 or 1). `SMOTENC` handles this correctly
by treating the specified columns as categorical (uses the mode of
neighbors instead of interpolating). We compute the one-hot column
positions for each variant's `tree`/`std`/`rob` feature matrix so
`training.ipynb` can pass them to `SMOTENC`.


In [8]:
onehot_prefixes = ('Business_Type_', 'Region_')

cat_indices_by_variant = {}
for name, v in variants.items():
    cols = v['X_train_tree'].columns.tolist()
    cat_idx = [i for i, c in enumerate(cols) if c.startswith(onehot_prefixes)]
    cat_indices_by_variant[name] = cat_idx
    print(f"{name:>12}: {len(cat_idx)} one-hot categorical columns out of {len(cols)} total")


        full: 19 one-hot categorical columns out of 40 total
  reduced_v1: 19 one-hot categorical columns out of 33 total
  reduced_v2: 19 one-hot categorical columns out of 31 total


## 8. Sanity Checks

In [9]:
for name, v in variants.items():
    for key in ['X_train_tree', 'X_train_std', 'X_train_rob', 'X_train_cb']:
        n_missing = v[key].isnull().sum().sum()
        assert n_missing == 0, f"{name}/{key} has {n_missing} missing values!"
print("No missing values in any variant — OK.")

# Confirm reduced_v2 really is VIF-clean via the built feature frame (not just the earlier check)
vif_check = compute_vif(variants['reduced_v2']['X_train_tree'][
    [c for c in reduced_v2_numeric]
].astype(float))
print(f"\nMax VIF in reduced_v2 (rebuilt from variant dict): {vif_check['VIF'].max():.2f}")
assert vif_check['VIF'].max() < 5, "reduced_v2 should be VIF-clean (<5)"


No missing values in any variant — OK.



Max VIF in reduced_v2 (rebuilt from variant dict): 1.76


## 9. Save Artifacts for `training.ipynb`

In [10]:
bundle = {
    'variants': variants,  # dict of {full, reduced_v1, reduced_v2} -> feature frames
    'cat_indices_by_variant': cat_indices_by_variant,
    'y_train': data1['y_train'], 'y_val': data1['y_val'], 'y_test': data1['y_test'],
    'cat_feature_names_cb': ['Business_Type', 'Region'],  # for CatBoost cat_features (by position, computed in training.ipynb)
    'vif_full': vif_full, 'vif_reduced_v1': vif_v1, 'vif_reduced_v2': vif_v2,
    'exact_dup_drops': EXACT_DUP_DROPS, 'revenue_size_drops': REVENUE_SIZE_DROPS,
}

joblib.dump(bundle, ARTIFACTS_DIR / 'batch2_data.joblib')
print("Saved:", ARTIFACTS_DIR / 'batch2_data.joblib')


Saved: C:\Users\Ali Ahmed\OneDrive - Alexandria National University\Desktop\Etax\use_case_1\ML_model\Models_Batch_2\artifacts\batch2_data.joblib


---
**Next:** `training.ipynb` runs two independent experiment tracks —
(1) imbalance-handling comparison (class weights / SMOTENC / random
over/under-sampling / combined) on the `full` feature set using each
model's Batch 1 best hyperparameters, and (2) feature-set comparison
(`full` vs `reduced_v1` vs `reduced_v2`) with no sampling — then
`evaluation.ipynb` compares everything against the Batch 1 baseline.
